# LeakAudit - 05 - Load D3 and audit the min-std window heuristic (E8)

**The controlled testbed lets us isolate the effect of the 2023 window rule.**

- D3 has no site confound, so any performance change here is due to the window-selection rule alone.
- We compare 'minstd' (the 2023 heuristic) against random / first / full windows, under naive and condition-independent splits.
- Hypothesis: leaks are stationary and non-leaks transient, so 'pick the calmest second' may be a selection bias rather than a physical gain.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation
cfg.apply_mpl_style()
from tqdm.auto import tqdm

M3 = io_utils.scan_d3()
assert len(M3) > 0, "D3 empty - run notebook 00 diagnostics first."
print(f"D3 files: {len(M3)}   (background {int(M3.is_bg.sum())})")
print(M3[~M3.is_bg].groupby(["modality", "label"]).size().to_string())

D3 files: 282   (background 2)
modality    label 
accel       leak      64
            noleak    16
hydrophone  leak      96
            noleak    24
pressure    leak      64
            noleak    16


## Load one clip of each kind

The hydrophone clips are headerless 32-bit little-endian PCM at 8 kHz, per the
provider's conversion script, and only the first 30 s is the signal of record.
Notebook **05b** verifies this; run it first if you have not.

In [3]:
# The raw format is fixed by the provider's conversion script: 32-bit
# little-endian PCM, mono, 8 kHz, truncated to the first 240 000 samples (30 s).
# See notebook 05b for the verification.
raws = M3[(M3.ext == ".raw") & (~M3.is_bg)]
x0, sr0 = io_utils.load_d3_signal(raws.iloc[0].to_dict())
print(f"raw  : {os.path.basename(raws.path.iloc[0])}")
print(f"       n={len(x0):,}  sr={sr0}  sec={len(x0)/sr0:.2f}  "
      f"range=[{x0.min():.4f}, {x0.max():.4f}]")
assert abs(len(x0)/sr0 - 30.0) < 1e-6, "raw clip is not 30 s - re-run notebook 05b"

csv0 = M3[(M3.ext == '.csv') & (~M3.is_bg)].iloc[0].to_dict()
xc, src = io_utils.load_d3_signal(csv0)
print(f"csv  : n={len(xc):,}  sr={src}  sec={len(xc)/src:.2f}")

raw  : BR_CC_0.18 LPS_N_H1.raw
       n=240,000  sr=8000  sec=30.00  range=[-0.5950, 0.6439]
csv  : n=923,488  sr=51200  sec=18.04


## Feature extraction for four window-selection modes

Cached per mode so re-runs are fast. Modality defaults to hydrophone.

In [4]:
MODALITY = "hydrophone"

def build_features(mode, sec=1.0, modality=MODALITY, seed=0):
    fp = cfg.CACHE / f"d3_{modality}_{mode}_{sec}.npz"
    if fp.exists():
        z = np.load(fp, allow_pickle=True)
        return z["X"]
    sub = M3[(~M3.is_bg) & (M3.modality == modality) & M3.label.notna()].reset_index(drop=True)
    Xs = []
    for i, r in tqdm(sub.iterrows(), total=len(sub), desc=f"{modality}/{mode}"):
        x, sr = io_utils.load_d3_signal(r.to_dict())
        xb = features.bandpass(x, sr)
        xw = features.pick_window(xb, sr, sec=sec, mode=mode, seed=seed + i)
        Xs.append(features.summary_features(xw, sr))
    X = np.vstack(Xs)
    np.savez_compressed(fp, X=X)
    return X

sub3 = M3[(~M3.is_bg) & (M3.modality == MODALITY) & M3.label.notna()].reset_index(drop=True)
y3   = sub3.label.to_numpy()
cond = (sub3.topology.astype(str) + "_" + sub3.leaktype.astype(str)).to_numpy()

X3 = {m: build_features(m) for m in ["minstd", "random", "first", "full"]}
for m, Xm in X3.items():
    print(f"  {m:8s} -> {Xm.shape}")

hydrophone/full: 100%|██████████| 120/120 [00:03<00:00, 36.25it/s]

  minstd   -> (120, 199)
  random   -> (120, 199)
  first    -> (120, 199)
  full     -> (120, 199)


## E8 - does the min-std heuristic help?

In [5]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
rows = []
for mode, Xm in X3.items():
    for seed in range(cfg.N_SEEDS):
        a, b = train_test_split(np.arange(len(y3)), test_size=.3,
                                random_state=seed, stratify=y3)
        rows.append(dict(mode=mode, split="naive", seed=seed,
                         **evaluation.evaluate(Xm[a], y3[a], Xm[b], y3[b], "gbdt", seed)))
        a, b = next(GroupShuffleSplit(1, test_size=.3, random_state=seed)
                    .split(np.arange(len(y3)), y3, groups=cond))
        if len(np.unique(y3[a])) > 1 and len(np.unique(y3[b])) > 1:
            rows.append(dict(mode=mode, split="condition_indep", seed=seed,
                             **evaluation.evaluate(Xm[a], y3[a], Xm[b], y3[b], "gbdt", seed)))

e8 = pd.DataFrame(rows)
s8 = e8.groupby(["mode", "split"]).agg(acc=("acc", "mean"), sd=("acc", "std"),
                                       base=("base", "mean"), n=("acc", "size")).reset_index()
print(s8.round(4).to_string(index=False))
e8.to_csv(cfg.RESULTS / "E8_window_heuristic_raw.csv", index=False)
s8.to_csv(cfg.RESULTS / "E8_window_heuristic_summary.csv", index=False)

piv = s8.pivot(index="mode", columns="split", values="acc")
if {"minstd", "random"} <= set(piv.index):
    print("\nminstd - random  (positive => the 2023 heuristic helps):")
    print((piv.loc["minstd"] - piv.loc["random"]).round(4).to_string())

  mode           split    acc     sd   base  n
 first condition_indep 0.6222 0.0421 0.6667  5
 first           naive 0.7806 0.0606 0.8056 10
  full condition_indep 0.6722 0.0124 0.6667  5
  full           naive 0.8194 0.0559 0.8056 10
minstd condition_indep 0.6167 0.0232 0.6667  5
minstd           naive 0.8083 0.0515 0.8056 10
random condition_indep 0.6333 0.0304 0.6667  5
random           naive 0.8000 0.0468 0.8056 10

minstd - random  (positive => the 2023 heuristic helps):
split
condition_indep   -0.0167
naive              0.0083
